# Human gastric cancer train

Set paths and samples in `config.yaml`, then run the cells in order.


## Settings


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / "experiments").is_dir() and (path / "lrpairs").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "experiments"))

from _shared.runtime import training_epochs, training_routes, scanvi_train_kwargs
from pathlib import Path
import yaml

DATASET = 'GP1'
CWD = Path.cwd().resolve()
EXPERIMENT_DIR = CWD if (CWD / "config.yaml").is_file() else CWD / "experiments" / "human_gastric_cancer"
REPO_ROOT = EXPERIMENT_DIR.parents[1]
import sys

CONFIG = yaml.safe_load(
    (EXPERIMENT_DIR / "config.yaml").read_text(encoding="utf-8")
)

def experiment_path(value):
    path = Path(value)
    return path if path.is_absolute() else (EXPERIMENT_DIR / path).resolve()

DATA_ROOT = experiment_path(CONFIG["data_root"])
RUN_ROOT = experiment_path(CONFIG["run_root"])
CHECKPOINT_ROOT = experiment_path(CONFIG["checkpoint_root"])
SCANVI_DIR = experiment_path(CONFIG["scanvi_dir"])
SCANVI_ADATA = SCANVI_DIR / "adata.h5ad"
STAGE1_CHECKPOINT_ROOT = experiment_path(CONFIG["stage1_checkpoint"])
STAGE2_CHECKPOINT_ROOT = experiment_path(CONFIG["stage2_checkpoint"])
LR_PAIRS = experiment_path(CONFIG["lr_pairs_path"])
TRANSITION_PRIOR = experiment_path(CONFIG["transition_prior_path"]) if "transition_prior_path" in CONFIG else None
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)



In [3]:
import scanpy as sc
import torch
import numpy as np
import pandas as pd
import sys

import scvi
from pathlib import Path
import importlib
from stvirtual.models import stage1_3d as s1
from stvirtual.models import stage2_3d_transition as s2


In [4]:
data_path = str(DATA_ROOT)
ckpt_path = str(CHECKPOINT_ROOT)
resl_path = str(RUN_ROOT)
lrpr_path = str(LR_PAIRS)
diff_path = str(TRANSITION_PRIOR)


In [5]:
route_ids=['normal', 'cancer']
fracs = ['normal_to_cancer']
seg_key = "normal_to_cancer" 
steps = 100


In [6]:
adata = sc.read_h5ad(SCANVI_ADATA)
adata


AnnData object with n_obs × n_vars = 391 × 36601
    obs: 'in_tissue', 'array_row', 'array_col', 'cluster', '_scvi_batch', '_scvi_labels', 'state', 'tissue_side', 'status', 'source', 'cx_aligned', 'cy_aligned'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'X_pca', 'X_scanVI', 'X_umap', 'spatial'
    varm: 'PCs'
    layers: 'counts'

In [7]:
adata.obs['status'].unique()


['normal', 'cancer']
Categories (2, object): ['cancer', 'normal']

In [8]:
model = None


## Train Stage 1


In [9]:
importlib.reload(s1)

res1 = s1.train_model_multislice(
    model=model,                  
    adata_all=adata,
    slice_key="status",
    route_ids=route_ids,
    save_root=str(STAGE1_CHECKPOINT_ROOT),
    x_key="cx_aligned",
    y_key="cy_aligned",
    latent_key=CONFIG["latent_key"],    
    steps=steps,  
    guide_eps=0.01,
    uot_eps=CONFIG["stage1_loss"]["uot_eps"], uot_tau=CONFIG["stage1_loss"]["uot_tau"], uot_lam_x=CONFIG["stage1_loss"]["uot_lam_x"], uot_lam_f=CONFIG["stage1_loss"]["uot_lam_f"],
    guide_topk=32, guide_temp=0.5, guide_schedule="linear",
    terrain_gap=False,
    cell_type_key='cluster',
    lam_context=0.0,
    lam_residual=1.0,
    lam_vsmooth=0.1,  
    lam_uot=CONFIG["stage1_loss"]["lam_uot"],
    lib_layer='counts',              
    latent_dim=10,             
    epochs=training_epochs(100),
    device="cuda:0",
)



[MultiSlice] Train segment: normal_to_cancer (n_src=145, n_tgt=246)
  Global norm route_ids=['normal', 'cancer']  latent_dim=10



Train (NeuralODE dopri5):   0%|          | 0/100 [00:00<?, ?it/s]

Train (NeuralODE dopri5):   0%|          | 0/100 [00:00<?, ?it/s, loss=0.1201, uot=0.1156, vs=0.0446]

Train (NeuralODE dopri5):   1%|          | 1/100 [00:00<01:29,  1.10it/s, loss=0.1201, uot=0.1156, vs=0.0446]

Train (NeuralODE dopri5):   1%|          | 1/100 [00:01<01:29,  1.10it/s, loss=0.1187, uot=0.1154, vs=0.0331]

...

Train (NeuralODE dopri5):  98%|█████████▊| 98/100 [00:32<00:00,  3.06it/s, loss=0.0850, uot=0.0677, vs=0.0407]

Train (NeuralODE dopri5):  99%|█████████▉| 99/100 [00:33<00:00,  3.11it/s, loss=0.0850, uot=0.0677, vs=0.0407]

Train (NeuralODE dopri5):  99%|█████████▉| 99/100 [00:33<00:00,  3.11it/s, loss=0.0846, uot=0.0675, vs=0.0443]

Train (NeuralODE dopri5): 100%|██████████| 100/100 [00:33<00:00,  3.11it/s, loss=0.0846, uot=0.0675, vs=0.0443]

Train (NeuralODE dopri5): 100%|██████████| 100/100 [00:33<00:00,  3.00it/s, loss=0.0846, uot=0.0675, vs=0.0443]

## Create boundaries


In [10]:
from stvirtual.utils import boundary_2d as boundary
from stvirtual.utils.stage1_results import save_res as save_stage1_results
from stvirtual.utils.trajectory import simulation_trace_from_out

boundary_cfg = CONFIG["boundary"]
saved = save_stage1_results(
    res1=res1, adata_all=adata,
    out_dir=str(CHECKPOINT_ROOT / "stage1_res"),
    slice_key="status", ann_key="cluster",
    save_prefix="simulation_stage1", steps=steps,
    n_cache=boundary_cfg["n_cache"], unnormalize=False,
)

for stage_index, stage_key in enumerate(fracs):
    trace = simulation_trace_from_out(res1[stage_key], steps=steps, n_cache=boundary_cfg["n_cache"], unnormalize=False)
    frames = [value.detach().cpu().numpy().astype(np.float32) for value in trace["x"]]
    stage_dir = RUN_ROOT / "bound" / stage_key
    stage_dir.mkdir(parents=True, exist_ok=True)
    statistics = []
    for frame_index, coordinates in enumerate(frames):
        shell, fraction, outside, expansion = boundary.make_shell_adaptive(
            coordinates, alpha_factor=boundary_cfg["alpha_factor"],
            seed=2026 + stage_index + frame_index,
            n_resample=boundary_cfg["n_resample"], target_frac=boundary_cfg["target_frac"],
            expand0=boundary_cfg["expand0"], expand_step=boundary_cfg["expand_step"],
            expand_max=boundary_cfg["expand_max"], fallback_expand=boundary_cfg["fallback_expand"],
        )
        boundary.save_shell_csv(shell, stage_dir / f"bound_z{frame_index:03d}.csv")
        boundary.plot_shell_coverage(coordinates, shell, outside, stage_dir / f"bound_z{frame_index:03d}.png")
        statistics.append({"frame": frame_index, "n_cells": len(coordinates), "fraction_inside": fraction, "n_outside": len(outside), "expansion": expansion})
    pd.DataFrame(statistics).to_csv(stage_dir / "bound_stats.csv", index=False)

print("Stage-1 traces:", saved)
print("Boundary root:", RUN_ROOT / "bound")


Stage-1 traces: {'normal_to_cancer': '/home/zhouyj/stVirtual_tutorial/experiments/human_gastric_cancer/artifacts/checkpoints/stage1_res/simulation_stage1_normal_to_cancer.npz'}
Boundary root: /home/zhouyj/stVirtual_tutorial/experiments/human_gastric_cancer/artifacts/results/bound


## Train Stage 2


In [11]:
importlib.reload(s2)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

ctx = s2.build_global_ctx(
    adata_path=str(SCANVI_ADATA),
    lr_pairs_path=lrpr_path,
    ckpt_3dslice=str(STAGE1_CHECKPOINT_ROOT / "normal_to_cancer" / "checkpoints" / "best.pt"),
    device=device,
    layer_col="cluster",
)

stages = []
for sk in fracs:
    src, tgt = sk.split("_to_", 1)
    stages.append(
        s2.StageCfg(
            src=src,
            tgt=tgt,
            out_npz_path=saved[sk],
            bound_dir=f"{resl_path}/bound/{sk}",
            decoder_checkpoint=str(experiment_path(CONFIG["decoder_checkpoint"].format(src=src, tgt=tgt))),
            scanvi_dir=None,
            diff_csv=diff_path,
            model_type="scanvi",
            use_lr=True,
            lr_source="decoder",
            use_latent=True,
            latent_key=CONFIG["latent_key"],
            z_csv_offset=0,
            layer_col="cluster",  
        )
    )

outs = s2.run_multi_stages(
    ctx=ctx,
    sample_key = 'status',
    stages=stages,
    best_ckpt_dir=str(STAGE2_CHECKPOINT_ROOT),
    train_kwargs=dict(EPOCHS=training_epochs(100), LR=1e-4),
    rl_xy=CONFIG["stage2_loss"]["rl_xy"],
    rl_z=CONFIG["stage2_loss"]["rl_z"],
)

for o in outs:
    print(o["best_ckpt_path"])


[Grid] H=32, W=32, H×W=1024
[auto cap] 1 {'src': {'q': 0.9, 'quantile_value': 1.0, 'mean': 1.0, 'max': 1.0, 'n_cells_used': 144}, 'tgt': {'q': 0.9, 'quantile_value': 1.0, 'mean': 1.0174672603607178, 'max': 2.0, 'n_cells_used': 229}, 'cap0': 1, 'capT': 1, 'cap': 1}


[diff] loaded /home/zhouyj/stVirtual_tutorial/experiments/human_gastric_cancer/data/diff_map.csv Kmax= 2


Training:   0%|          | 0/100 [00:00<?, ?it/s]

Training:   0%|          | 0/100 [00:05<?, ?it/s, rew=-0.5538, best=-0.5538, tgt=0.4605, occ=0.8966]

Training:   1%|          | 1/100 [00:05<09:53,  6.00s/it, rew=-0.5538, best=-0.5538, tgt=0.4605, occ=0.8966]

...

Training:  99%|█████████▉| 99/100 [09:42<00:05,  5.80s/it, rew=-0.5362, best=-0.4731, tgt=0.4419, occ=0.9052]

Training:  99%|█████████▉| 99/100 [09:48<00:05,  5.80s/it, rew=-0.5485, best=-0.4731, tgt=0.4585, occ=0.8621]

Training: 100%|██████████| 100/100 [09:48<00:00,  5.81s/it, rew=-0.5485, best=-0.4731, tgt=0.4585, occ=0.8621]

Training: 100%|██████████| 100/100 [09:48<00:00,  5.88s/it, rew=-0.5485, best=-0.4731, tgt=0.4585, occ=0.8621]

/home/zhouyj/stVirtual_tutorial/experiments/human_gastric_cancer/artifacts/checkpoints/stage2/policy_normal_to_cancer.pt


In [12]:
ckpt_map = {}
for o in outs:
    cfg = o["stage"]
    ckpt_map[f"{cfg.src}_to_{cfg.tgt}"] = o["best_ckpt_path"]

ckpt_map


{'normal_to_cancer': '/home/zhouyj/stVirtual_tutorial/experiments/human_gastric_cancer/artifacts/checkpoints/stage2/policy_normal_to_cancer.pt'}

## Run simulation


In [13]:
importlib.reload(s2)
simulations = {}
for cfg in stages:
    seg_key = f"{cfg.src}_to_{cfg.tgt}"
    simulations[seg_key] = s2.simulation_policy_one_stage(
        s2, ctx, cfg, ckpt_map[seg_key],sample_key='status',
        seed=2026, ADVECT_LATENT=True,
        output_dir=Path(resl_path) / 'simulation' / seg_key, output_prefix=seg_key,  
    )
    print(seg_key, "Tp1=", len(simulations[seg_key]["coords"]))


[Grid] H=32, W=32, H×W=1024
[auto cap] 1 {'src': {'q': 0.9, 'quantile_value': 1.0, 'mean': 1.0, 'max': 1.0, 'n_cells_used': 144}, 'tgt': {'q': 0.9, 'quantile_value': 1.0, 'mean': 1.0174672603607178, 'max': 2.0, 'n_cells_used': 229}, 'cap0': 1, 'capT': 1, 'cap': 1}


[diff] loaded /home/zhouyj/stVirtual_tutorial/experiments/human_gastric_cancer/data/diff_map.csv Kmax= 2


normal_to_cancer Tp1= 101


In [14]:
TRANSITION_OBS_FIELDS = (
    "uid",
    "parent_uid",
    "diff_alpha",
    "src_layer_id",
    "src_layer",
    "tgt_layer_id",
    "tgt_layer",
)

def persist_transition_simulation_obs(simulation_result, celltype_names):
    output_paths = simulation_result.get("output_paths")
    required = ("uid", "parent_uid", "layers", "is_diff", "diff_alpha", "diff_tgt_layer", "commit_step")
    missing = [field for field in required if not isinstance(simulation_result.get(field), list)]
    if missing:
        raise KeyError(f"simulation_result is missing transition fields: {missing}")
    if not isinstance(output_paths, list):
        raise KeyError("simulation_result has no output_paths; pass output_dir to simulation_policy_one_stage")
    frame_count = len(simulation_result["coords"])
    if len(output_paths) != frame_count:
        raise ValueError("output_paths and simulation frames have different lengths")

    names = [str(name) for name in celltype_names]
    source_by_uid = {}
    target_by_uid = {}
    for frame_index in range(frame_count):
        uids = np.asarray(simulation_result["uid"][frame_index], dtype=np.int64)
        current_layers = np.asarray(simulation_result["layers"][frame_index], dtype=np.int64)
        target_layers = np.asarray(simulation_result["diff_tgt_layer"][frame_index], dtype=np.int64)
        for uid, current_layer, target_layer in zip(uids, current_layers, target_layers):
            source_by_uid.setdefault(int(uid), int(current_layer))
            if int(target_layer) >= 0:
                target_by_uid[int(uid)] = int(target_layer)

    for frame_index, output_path in enumerate(output_paths):
        frame_adata = sc.read_h5ad(output_path)
        uids = np.asarray(simulation_result["uid"][frame_index], dtype=np.int64)
        parent_uids = np.asarray(simulation_result["parent_uid"][frame_index], dtype=np.int64)
        diff_alpha = np.asarray(simulation_result["diff_alpha"][frame_index], dtype=np.float32)
        is_diff = np.asarray(simulation_result["is_diff"][frame_index], dtype=bool)
        commit_step = np.asarray(simulation_result["commit_step"][frame_index], dtype=np.int32)
        has_transition = is_diff | (commit_step >= 0)

        src_ids = np.full(frame_adata.n_obs, -1, dtype=np.int64)
        tgt_ids = np.full(frame_adata.n_obs, -1, dtype=np.int64)
        for index, uid in enumerate(uids):
            if has_transition[index]:
                src_ids[index] = source_by_uid[int(uid)]
                tgt_ids[index] = target_by_uid.get(int(uid), -1)

        src_names = np.array([names[index] if 0 <= index < len(names) else "" for index in src_ids], dtype=str)
        tgt_names = np.array([names[index] if 0 <= index < len(names) else "" for index in tgt_ids], dtype=str)
        frame_adata.obs["uid"] = uids
        frame_adata.obs_names = uids.astype(str)
        frame_adata.obs["parent_uid"] = parent_uids
        frame_adata.obs["diff_alpha"] = diff_alpha
        frame_adata.obs["src_layer_id"] = src_ids
        frame_adata.obs["src_layer"] = src_names
        frame_adata.obs["tgt_layer_id"] = tgt_ids
        frame_adata.obs["tgt_layer"] = tgt_names
        frame_adata.write_h5ad(output_path, compression="gzip")

    return {"frames_updated": frame_count, "obs_fields": list(TRANSITION_OBS_FIELDS)}

simulation_obs_reports = {
    route: persist_transition_simulation_obs(route_simulation, ctx.layers_list)
    for route, route_simulation in simulations.items()
}
simulation_obs_reports


{'normal_to_cancer': {'frames_updated': 101,
  'obs_fields': ['uid',
   'parent_uid',
   'diff_alpha',
   'src_layer_id',
   'src_layer',
   'tgt_layer_id',
   'tgt_layer']}}

In [15]:
r0 = simulations['normal_to_cancer']


In [16]:
import numpy as np
import pandas as pd

fi = 1           

layers = np.asarray(r0["layers"][fi]).astype(int)
is_diff = np.asarray(r0["is_diff"][fi]).astype(bool)
diff_tgt = np.asarray(r0["diff_tgt_layer"][fi]).astype(int)
commit = np.asarray(r0["commit_step"][fi]).astype(int)

m = is_diff & (diff_tgt >= 0) & (commit < 0)

df = pd.DataFrame({
    "src_layer_internal": layers[m],
    "tgt_layer_internal": diff_tgt[m],
})

print("n_active_diff =", len(df))
print(df.value_counts().sort_index())


n_active_diff = 25
src_layer_internal  tgt_layer_internal
1                   2                     5
                    5                     4
3                   2                     5
                    5                     2
4                   2                     4
                    5                     5
Name: count, dtype: int64


In [17]:
ro = simulations[seg_key]

for t in range(len(ro["coords"])):
    N  = int(np.asarray(ro["coords"][t]).shape[0])
    nb = int(np.asarray(ro["is_birth"][t]).sum())  
    print(f"{seg_key} | t={t:02d} | N={N} | new_birth={nb}")


normal_to_cancer | t=00 | N=145 | new_birth=0
normal_to_cancer | t=01 | N=146 | new_birth=1
normal_to_cancer | t=02 | N=147 | new_birth=1
normal_to_cancer | t=03 | N=148 | new_birth=1
normal_to_cancer | t=04 | N=149 | new_birth=1
normal_to_cancer | t=05 | N=150 | new_birth=2
normal_to_cancer | t=06 | N=151 | new_birth=15
normal_to_cancer | t=07 | N=152 | new_birth=17
normal_to_cancer | t=08 | N=153 | new_birth=17
normal_to_cancer | t=09 | N=154 | new_birth=8
normal_to_cancer | t=10 | N=155 | new_birth=1
normal_to_cancer | t=11 | N=156 | new_birth=2
normal_to_cancer | t=12 | N=157 | new_birth=1
normal_to_cancer | t=13 | N=158 | new_birth=6
normal_to_cancer | t=14 | N=159 | new_birth=13
normal_to_cancer | t=15 | N=160 | new_birth=23
normal_to_cancer | t=16 | N=161 | new_birth=22
normal_to_cancer | t=17 | N=162 | new_birth=26
normal_to_cancer | t=18 | N=163 | new_birth=27
normal_to_cancer | t=19 | N=164 | new_birth=25
normal_to_cancer | t=20 | N=165 | new_birth=26
normal_to_cancer | t=21 

In [18]:
ro.keys()


dict_keys(['coords', 'layers', 'latent', 'lr', 'anchor', 'is_birth', 'born_step', 'uid', 'parent_uid', 'event', 'is_diff', 'diff_alpha', 'diff_alpha_post', 'diff_tgt_layer', 'diff_enter_step', 'commit_step', 't', 'T', 'n_layers', 'output_paths'])

In [19]:
from _shared.runtime import save_run_gallery

overview_path = save_run_gallery(EXPERIMENT_DIR)


Saved 22 panels: /home/zhouyj/stVirtual_tutorial/experiments/human_gastric_cancer/artifacts/results/overview/normal_to_cancer.png
